# CUM Series 15 — Universal Muon: Kill the AdamW Crutch

Phase 1: Single optimizer for ALL parameter shapes.
- **2D params**: NS orthogonalization (same as Muon)
- **1D params**: Unit normalization (polar factor of a vector)
- **Scalars**: Raw gradient (SGD)

Key hypothesis: Unified convergence dynamics > split optimizer.

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        DATA_PATH)
with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum.tensor import UniversalMuon

MODEL_CFG = dict(d_model=128, n_heads=4, n_layers=4, d_ff=512, ctx_len=256)
BATCH_SIZE = 32
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250
BASE_LR = 0.02
SEED = 42

test_model = MicroGPT(vocab_size=65, **MODEL_CFG)
print(f'Model: {sum(p.numel() for p in test_model.parameters())/1e6:.1f}M params')
del test_model
print('Imports OK')

In [ ]:
class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y

dataset = CharDataset(TEXT, ctx_len=MODEL_CFG['ctx_len'], device=DEVICE)
print(f'Dataset on {DEVICE}: vocab={dataset.vocab_size}')

In [ ]:
class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def print_param_breakdown(model, cfg_name):
    """Print how many params are in each shape group."""
    counts = {0: 0, 1: 0, 2: 0}
    nums = {0: 0, 1: 0, 2: 0}
    for p in model.parameters():
        d = min(p.ndim, 2)
        counts[d] += 1
        nums[d] += p.numel()
    total = sum(nums.values())
    print(f'  [{cfg_name}] Param breakdown:')
    for d in [2, 1, 0]:
        label = {0: 'scalar', 1: '1D', 2: '2D'}[d]
        if counts[d] > 0:
            print(f'    {label}: {counts[d]} params, {nums[d]:,} elements ({100*nums[d]/total:.1f}%)')
    print(f'    total: {sum(counts.values())} params, {total:,} elements')


def train_one(name, optimizers, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    """Train with a list of optimizers."""
    model.train()
    trajectory = []

    # Warmup iterations for torch.compile
    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        for opt in optimizers:
            opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        for opt in optimizers:
            for g in opt.param_groups:
                if isinstance(opt, torch.optim.AdamW):
                    g['lr'] = get_lr(step, total_steps, warmup_steps, 3e-4)
                else:
                    g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        for opt in optimizers:
            opt.zero_grad()
        loss.backward()
        for opt in optimizers:
            opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
def make_model_and_opts(dataset, cfg):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **MODEL_CFG).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')

    t = cfg['type']

    if t == 'Muon+AdamW':
        hidden_2d = model.get_hidden_2d_params()
        other = model.get_other_params()
        muon = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
        adam = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)
        optimizers = [muon, adam]

    elif t == 'Universal all':
        uni = UniversalMuon(
            model.parameters(),
            lr=BASE_LR,
            beta1=0.95,
            ns_steps=5,
            scale_1d=cfg.get('scale_1d', 1.0),
        )
        optimizers = [uni]

    elif t == 'Universal 2D + AdamW 1D':
        params_2d = [p for p in model.parameters() if p.ndim == 2]
        params_1d = [p for p in model.parameters() if p.ndim < 2]
        uni = UniversalMuon(params_2d, lr=BASE_LR, ns_steps=5)
        adam = torch.optim.AdamW(params_1d, lr=3e-4, weight_decay=0.01)
        optimizers = [uni, adam]

    else:
        raise ValueError(f'Unknown: {t}')

    return model, optimizers


def run_all(configs):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, optimizers = make_model_and_opts(dataset, cfg)
            print_param_breakdown(model._orig_mod if hasattr(model, '_orig_mod') else model, name)
            val_loss, traj, elapsed = train_one(name, optimizers, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='15a'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon+AdamW'), None)

    print(f'\n## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon+AdamW | Time |')
    print(f'|--------|----------|---------------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r.get('error'):
            print(f'| {r["name"]} | FAILED | -- | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '--'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {r["elapsed"]:.0f}s |')


print('Factory ready')

---
## 15a: Phase 1 — Basic Universal Muon

Can a single optimizer with shape-aware updates match/beat Muon+AdamW?

In [ ]:
CONFIGS_15A = [
    {'name': 'Muon+AdamW (baseline)', 'type': 'Muon+AdamW'},
    {'name': 'Universal all s1d=1.0', 'type': 'Universal all', 'scale_1d': 1.0},
    {'name': 'Universal all s1d=0.015', 'type': 'Universal all', 'scale_1d': 0.015},
    {'name': 'Universal 2D + AdamW 1D', 'type': 'Universal 2D + AdamW 1D'},
]

results_15a = run_all(CONFIGS_15A)
show_results(results_15a, '15a: Phase 1 Universal Muon')